# Olist | Crescimento e Receita
### Diagnóstico comercial para acionistas e investidores
**Pergunta de investimento:** o crescimento é consistente? Quais categorias, regiões, produtos e sellers concentram valor — e onde priorizar expansão?

**Trilha:** qualidade → KPIs e série temporal → mix de categorias → geografia → líderes comerciais → síntese acionável.

> **Nota de interpretação:** GMV dos itens entregues é um indicador de vendas brutas de produtos, **não é faturamento líquido da plataforma nem lucro**. Todos os gráficos e conclusões são produzidos a partir dos CSVs reais quando o notebook é executado.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="talk", font_scale=0.83)
plt.rcParams.update({"figure.figsize": (12, 6), "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "axes.titlesize": 17, "axes.titleweight": "bold"})
NAVY = "#162D4C"
BLUE = "#2B6CB0"
TEAL = "#12A89D"
ORANGE = "#F59E50"
GREY = "#728096"

def locate_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw").exists():
            return base
    raise FileNotFoundError("Pasta data/raw não encontrada. Abra o notebook dentro da estrutura do projeto.")

ROOT = locate_root()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=190, bbox_inches="tight", facecolor="white")
    plt.show()

def brl(v):
    return f"R$ {v:,.2f}".replace(",", "@").replace(".", ",").replace("@", ".")

def short_brl(v):
    return f"R$ {v/1e6:,.1f} mi" if abs(v) >= 1e6 else (f"R$ {v/1e3:,.1f} mil" if abs(v)>=1e3 else brl(v))

def pct(v):
    return f"{100*v:.1f}%".replace(".", ",")

## 1. Leitura das bases e governança
Usamos apenas 6 arquivos do conjunto original. Chaves são validadas antes das junções para evitar duplicações inadvertidas de receita.

In [ ]:
files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "translations": "product_category_name_translation.csv",
    "sellers": "olist_sellers_dataset.csv",
    "customers": "olist_customers_dataset.csv",
}
# Algumas distribuições utilizam o prefixo olist_ também na tradução
translation_alt = RAW / "olist_product_category_name_translation.csv"
paths = {key: RAW / file for key, file in files.items()}
if not paths["translations"].exists() and translation_alt.exists():
    paths["translations"] = translation_alt
missing = [path.name for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"CSV(s) não encontrado(s) em {RAW}: {missing}. Consulte o README.md.")
dfs = {key: pd.read_csv(path) for key, path in paths.items()}
for key, frame in dfs.items():
    print(f"{key:14s} | {len(frame):>8,} linhas | {frame.shape[1]:>2} colunas | duplicadas exatas: {frame.duplicated().sum():,}")

for name, key in [("orders", "order_id"), ("products", "product_id"),
                  ("sellers", "seller_id"), ("customers", "customer_id")]:
    assert dfs[name][key].is_unique, f"Chave não única: {name}.{key}"
assert not dfs['items'].duplicated(['order_id', 'order_item_id']).any(), "Itens com chave duplicada"
print("Validação de chaves: OK")

In [ ]:
orders = dfs["orders"][["order_id", "customer_id", "order_status", "order_purchase_timestamp"]].copy()
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
orders = orders.loc[(orders["order_status"] == "delivered") & orders["order_purchase_timestamp"].notna()]
items = dfs["items"][["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"]].copy()
items["price"] = pd.to_numeric(items["price"], errors="coerce")
items = items.loc[items["price"].notna() & (items["price"] >= 0)]
products = dfs["products"][["product_id", "product_category_name"]].copy()
translations = dfs["translations"][["product_category_name", "product_category_name_english"]].drop_duplicates('product_category_name')
products = products.merge(translations, on="product_category_name", how="left", validate="many_to_one")
products["categoria"] = products["product_category_name_english"].fillna(products["product_category_name"]).fillna("Sem categoria")
sellers = dfs["sellers"][["seller_id", "seller_city", "seller_state"]].copy()
customers = dfs["customers"][["customer_id", "customer_city", "customer_state"]].copy()

base = (items.merge(orders, on="order_id", how="inner", validate="many_to_one")
             .merge(products[["product_id", "categoria"]], on="product_id", how="left", validate="many_to_one")
             .merge(sellers, on="seller_id", how="left", validate="many_to_one")
             .merge(customers, on="customer_id", how="left", validate="many_to_one"))
base["categoria"] = base["categoria"].fillna("Sem categoria")
base["uf_cliente"] = base["customer_state"].fillna("NI")
base["municipio_cliente"] = base["customer_city"].fillna("Não informado")
base["uf_seller"] = base["seller_state"].fillna("NI")
base["mes"] = base["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
assert base[["order_id", "order_item_id"]].duplicated().sum() == 0, "Chave item multiplicada"
print("Pedidos entregues na base analítica:", f"{base.order_id.nunique():,}")
print("Itens:", f"{len(base):,}")
print("Intervalo de compra:", base.order_purchase_timestamp.min().date(), "a", base.order_purchase_timestamp.max().date())
print("Receita bruta de produtos:", brl(base.price.sum()))
display(pd.DataFrame({"campo": base.columns, "percentual_nulos": (base.isna().mean()*100).round(2).values}).query("percentual_nulos > 0"))

## 2. Visão executiva | Crescimento e Receita
**Três KPIs consistentes:** pedidos entregues, GMV de produtos e ticket médio. Meses de fronteira podem ter cobertura parcial; a variação mês a mês deve ser lida com cautela.

In [ ]:
monthly = (base.groupby("mes").agg(receita=("price", "sum"), pedidos=("order_id", "nunique"))
           .sort_index())
monthly["ticket_medio"] = monthly["receita"] / monthly["pedidos"]
monthly["variacao_receita_pct"] = monthly["receita"].pct_change() * 100
monthly["variacao_pedidos_pct"] = monthly["pedidos"].pct_change() * 100
kpis = {"receita": base.price.sum(), "pedidos": base.order_id.nunique(),
        "ticket": base.price.sum()/base.order_id.nunique(), "itens": len(base)}
display(pd.DataFrame({"KPI": ["GMV produtos entregues", "Pedidos entregues", "Ticket médio por pedido", "Itens vendidos"],
                      "Valor": [brl(kpis["receita"]), f'{kpis["pedidos"]:,}', brl(kpis["ticket"]), f'{kpis["itens"]:,}']}))

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True, constrained_layout=True)
series = [("pedidos", "Pedidos entregues", BLUE), ("receita", "Receita de produtos (R$)", TEAL),
          ("ticket_medio", "Ticket médio (R$)", ORANGE)]
for ax, (key, label, color) in zip(axes, series):
    ax.plot(monthly.index, monthly[key], color=color, lw=2.7, marker="o", markersize=3)
    ax.fill_between(monthly.index, monthly[key].values, alpha=.12, color=color)
    ax.set_ylabel(label)
    ax.grid(axis="x", visible=False)
    ax.tick_params(axis='x', rotation=45)
fig.suptitle("Trajetória comercial | Volume, valor e monetização", x=.06, ha='left', fontsize=20, fontweight='bold', color=NAVY)
save(fig, "01_evolucao_mensal")
display(monthly.tail(8).style.format({"receita": brl, "ticket_medio": brl,
                                     "variacao_receita_pct": "{:.1f}%", "variacao_pedidos_pct": "{:.1f}%"}))

### Crescimento sem confundir tendência com sazonalidade
Comparação ano contra ano **apenas para meses presentes em ambos os anos**, evitando comparar um mês com outro de sazonalidade diferente. A cobertura de 2018 pode ser parcial.

In [ ]:
yoy = monthly[['receita','pedidos','ticket_medio']].copy()
yoy['ano'] = yoy.index.year
yoy['numero_mes'] = yoy.index.month
yoy['receita_ano_anterior'] = yoy.groupby('numero_mes')['receita'].shift(1)
yoy['ano_anterior_disponivel'] = yoy.groupby('numero_mes')['ano'].shift(1)
yoy['yoy_receita_pct'] = np.where(yoy['ano']-yoy['ano_anterior_disponivel']==1,
                                 100*(yoy['receita']/yoy['receita_ano_anterior']-1), np.nan)
yoy_valid = yoy.dropna(subset=['yoy_receita_pct'])
fig, ax = plt.subplots(figsize=(13, 5))
if not yoy_valid.empty:
    bars = ax.bar(yoy_valid.index.strftime('%Y-%m'), yoy_valid['yoy_receita_pct'],
                  color=np.where(yoy_valid['yoy_receita_pct']>=0, TEAL, ORANGE))
    ax.axhline(0, color=NAVY, linewidth=1)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('Variação YoY da receita (%)')
else:
    ax.text(.5, .5, 'Sem meses comparáveis consecutivos', transform=ax.transAxes, ha='center')
ax.set_title('Crescimento interanual | Mesmo mês contra ano anterior', color=NAVY, loc='left')
save(fig, '02_crescimento_interanual')

## 3. Quais categorias sustentam a receita?
Analisamos receita absoluta e concentração (participação das 10 maiores categorias). A métrica é venda dos itens, sem duplicação de frete.

In [ ]:
cat = (base.groupby('categoria').agg(receita=('price','sum'), pedidos=('order_id','nunique'),
                                      itens=('price','size')).sort_values('receita', ascending=False))
cat['participacao'] = cat['receita']/kpis['receita']
cat['ticket_item'] = cat['receita']/cat['itens']
top_cat = cat.head(10).sort_values('receita')
fig, ax = plt.subplots(figsize=(12, 7))
colors = [BLUE]*(len(top_cat)-1) + [TEAL]
ax.barh(top_cat.index, top_cat.receita/1e6, color=colors)
for y, (_, row) in enumerate(top_cat.iterrows()):
    ax.text(row.receita/1e6, y, f"  {pct(row.participacao)}", va='center', fontsize=10)
ax.set_xlabel('GMV de produtos (R$ milhões)')
ax.set_title('Categorias líderes | Receita e participação no total', loc='left', color=NAVY)
save(fig, '03_top_categorias')

fig, ax = plt.subplots(figsize=(12, 4.5))
cut = cat.head(10).participacao.sum()
ax.barh(['Mix de categorias'], [cut*100], color=TEAL, label='Top 10')
ax.barh(['Mix de categorias'], [(1-cut)*100], left=[cut*100], color='#CDD6E1', label='Demais')
ax.text(cut*50, 0, pct(cut), ha='center', va='center', color='white', fontweight='bold')
ax.set_xlim(0,100); ax.set_xlabel('Participação do GMV (%)'); ax.legend(loc='upper right')
ax.set_title('Concentração de receita | 10 principais categorias', loc='left', color=NAVY)
save(fig, '04_concentracao_categorias')

## 4. Onde estão os compradores?
A análise geográfica considera o **endereço cadastral do cliente** (demanda), não a localização do seller (oferta). Municípios homônimos são distinguidos por UF.

In [ ]:
uf = base.groupby('uf_cliente').agg(receita=('price','sum'), pedidos=('order_id','nunique')).sort_values('receita',ascending=False)
uf['participacao'] = uf['receita']/kpis['receita']
municipio = (base.groupby(['uf_cliente','municipio_cliente'])
             .agg(receita=('price','sum'),pedidos=('order_id','nunique')).sort_values('receita',ascending=False))
fig, axes = plt.subplots(1,2,figsize=(16,7), constrained_layout=True)
u = uf.head(10).sort_values('receita')
axes[0].barh(u.index, u.receita/1e6, color=BLUE)
axes[0].set_xlabel('GMV (R$ mi)'); axes[0].set_title('Top 10 UFs | Consumo',loc='left',color=NAVY)
m = municipio.head(10).reset_index()
m['local'] = m['municipio_cliente'].str.title() + ' / ' + m['uf_cliente']
m = m.sort_values('receita')
axes[1].barh(m['local'], m['receita']/1e6, color=TEAL)
axes[1].set_xlabel('GMV (R$ mi)'); axes[1].set_title('Top 10 municípios | Consumo',loc='left',color=NAVY)
save(fig, '05_geografia_uf_municipio')

uf_seller = base.groupby('uf_seller').agg(receita=('price','sum'), sellers=('seller_id','nunique')).sort_values('receita',ascending=False)
fig, ax = plt.subplots(figsize=(12,5))
s = uf_seller.head(10).sort_values('receita')
ax.barh(s.index, s.receita/1e6, color=ORANGE)
ax.set_xlabel('GMV de vendedores por UF (R$ mi)')
ax.set_title('Origem da oferta | UFs dos sellers',loc='left',color=NAVY)
save(fig, '06_origem_sellers')

## 5. Quais produtos e sellers são top performers?
Usamos receita, pedidos e itens para distinguir liderança comercial de preço elevado. IDs podem ser usados como chave para enriquecer com nomes em um catálogo externo.

In [ ]:
prod = (base.groupby(['product_id','categoria']).agg(receita=('price','sum'),pedidos=('order_id','nunique'),
                                                        unidades=('price','size'))
        .sort_values('receita',ascending=False))
prod['participacao'] = prod['receita']/kpis['receita']
seller = (base.groupby(['seller_id','seller_city','seller_state'],dropna=False)
          .agg(receita=('price','sum'),pedidos=('order_id','nunique'), unidades=('price','size'))
          .sort_values('receita',ascending=False))
seller['participacao'] = seller['receita']/kpis['receita']

fig, axes = plt.subplots(1,2,figsize=(17,7),constrained_layout=True)
p=prod.head(10).reset_index().sort_values('receita')
axes[0].barh(p.product_id.str[:9]+'…',p.receita/1000,color=TEAL)
axes[0].set_title('Top 10 produtos por GMV',loc='left',color=NAVY)
axes[0].set_xlabel('GMV (R$ mil)')
s=seller.head(10).reset_index().sort_values('receita')
axes[1].barh(s.seller_id.str[:9]+'…',s.receita/1000,color=BLUE)
axes[1].set_title('Top 10 sellers por GMV',loc='left',color=NAVY)
axes[1].set_xlabel('GMV (R$ mil)')
save(fig,'07_top_produtos_sellers')

seller_sorted = seller.receita.sort_values(ascending=False).reset_index(drop=True)
seller_cumulative = seller_sorted.cumsum()/seller_sorted.sum()
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(np.arange(1,len(seller_cumulative)+1)/len(seller_cumulative)*100, seller_cumulative*100,color=BLUE,lw=2.7)
ax.plot([0,100],[0,100],ls='--',color=GREY,alpha=.8)
ax.set(xlabel='% acumulado de sellers (ordenados por receita)',ylabel='% acumulado do GMV',xlim=(0,100),ylim=(0,100))
ax.set_title('Concentração comercial | Curva de Pareto dos sellers',loc='left',color=NAVY)
save(fig,'08_pareto_sellers')
print('Top produtos'); display(prod.head(10))
print('Top sellers'); display(seller.head(10))

## 6. Síntese para o comitê de investimentos
Os números abaixo são calculados diretamente das bases. As ações indicadas são **hipóteses gerenciais**, não conclusões causais; cada uma deve ter meta e acompanhamento.

In [ ]:
top_uf = uf.index[0]
top_cat_name = cat.index[0]
top_city = municipio.index[0]
seller_n = len(seller)
top_10pct_n = max(1,int(np.ceil(seller_n*0.1)))
seller_top_decile = seller_sorted.head(top_10pct_n).sum()/seller_sorted.sum()
first12 = monthly.head(min(12,len(monthly)))
last12 = monthly.tail(min(12,len(monthly)))
summary = f"""# Resumo executivo | Olist Crescimento e Receita

## Leitura do negócio
- **GMV dos produtos entregues:** {brl(kpis['receita'])}.
- **Pedidos entregues com itens:** {kpis['pedidos']:,}.
- **Ticket médio:** {brl(kpis['ticket'])}.
- **Categoria de maior receita:** {top_cat_name}, com {pct(cat.iloc[0].participacao)} do GMV.
- **Participação das 10 maiores categorias:** {pct(cut)} do GMV.
- **Principal UF de consumo:** {top_uf}, com {pct(uf.iloc[0].participacao)} do GMV.
- **Principal município de consumo:** {top_city[1].title()} / {top_city[0]}.
- **10% de sellers com maior GMV:** concentram {pct(seller_top_decile)} das vendas dos sellers analisados.

## Recomendações orientadas por evidência
1. **Defender e desenvolver as categorias líderes.** Avaliar disponibilidade, mix e campanhas nas categorias mais representativas, acompanhando receita e conversão antes de ampliar orçamento. Categoria principal: {top_cat_name}.
2. **Fazer testes regionais.** Priorizar análise de potencial em UFs relevantes e comparar crescimento incremental por região; {top_uf} lidera a receita observada, mas liderança não prova maior retorno marginal.
3. **Gerir concentração de sellers.** Criar plano de continuidade e diversificação da oferta se dependência dos maiores vendedores representar risco; top 10% concentram {pct(seller_top_decile)} do GMV.
4. **Acelerar produtos vencedores com proteção de margem.** Testar melhor posicionamento dos top produtos e medir conversão, recompra e margem antes de escalar.
5. **Instituir governança mensal.** Monitorar pedidos, GMV, ticket e composição por categoria e UF; validar meses parciais e alinhar métricas ao financeiro.

## Limitações
- O GMV aqui é soma do preço dos itens de pedidos entregues por data de compra; exclui frete, cancelados e não entregues.
- Base histórica Olist, sem leitura da performance atual e sem apuração de lucro, margem, comissão, taxa de devolução ou CAC.
- Períodos de fronteira do dataset podem ser incompletos; não extrapolar como previsão.
"""
(OUT/'resumo_executivo.md').write_text(summary,encoding='utf-8')
with pd.ExcelWriter(OUT/'tabelas_analiticas.xlsx',engine='openpyxl') as writer:
    monthly.to_excel(writer,sheet_name='Mensal')
    cat.to_excel(writer,sheet_name='Categorias')
    uf.to_excel(writer,sheet_name='UF_Clientes')
    municipio.to_excel(writer,sheet_name='Municipios')
    uf_seller.to_excel(writer,sheet_name='UF_Sellers')
    prod.head(100).to_excel(writer,sheet_name='Top_100_Produtos')
    seller.head(100).to_excel(writer,sheet_name='Top_100_Sellers')
display(Markdown(summary))
print('Arquivos salvos em:', OUT)

## 7. Storytelling sugerido para apresentação

1. **Tese:** qual é o tamanho e a trajetória do comércio observado? Gráfico 01.
2. **Qualidade do crescimento:** volume versus ticket; teste da variação interanual. Gráfico 02.
3. **Motores de receita:** categorias líderes e concentração. Gráficos 03–04.
4. **Mapa de oportunidades:** demanda por UF e município versus oferta por seller. Gráficos 05–06.
5. **Dependências e execução:** principais produtos/sellers e concentração. Gráficos 07–08.
6. **Decisões para o próximo trimestre:** testes comerciais, regionais e governança de risco com KPIs de acompanhamento.

**Pergunta para decisão:** onde alocar recursos para crescer sem ampliar excessivamente concentração e risco operacional?

## 8. Contexto externo | Olist versus e-commerce brasileiro

**Nova base:** Webshoppers (Ebit/Nielsen), cifras anuais de 2017 e 2018. Preservamos a série interna da Olist, que mede **GMV dos itens entregues, sem frete**, e separamos claramente os conceitos de **faturamento setorial** e **GMV da amostra**. A comparação é **contextual, não uma estimativa de market share**. Para Olist usamos janeiro–agosto em ambos os anos, pois a parte final de 2018 não cobre ano fechado; o indicador setorial compara anos completos. **Não calcular diferença entre taxas como ganho de participação de mercado.**

Fonte do indicador setorial: Webshoppers/Ebit-Nielsen, divulgado em 25/03/2019: https://www.terra.com.br/byte/receita-do-e-commerce-no-brasil-foi-de-r-532-bi-em-2018-diz-estudo,586591935998801e529b01f0566a08fdt0ugp598.html . Dados transcritos e versionados em `data/external/`.

In [ ]:
# EXTENSÃO EXTERNA: taxas e comparação contextual
market = pd.read_csv(ROOT/'data'/'external'/'webshoppers_ebit_mercado_2017_2018.csv')
assert set(market['ano']) == {2017, 2018}
market_growth = market.set_index('ano')['faturamento_brl_bilhoes'].pct_change().loc[2018] * 100
# Mesmo conjunto de meses em ambos os anos; não usar 2018 set-dez nem 2016 parcial.
m_2017 = monthly.loc[(monthly.index.year==2017) & (monthly.index.month<=8)]
m_2018 = monthly.loc[(monthly.index.year==2018) & (monthly.index.month<=8)]
assert set(m_2017.index.month)==set(range(1,9)) and set(m_2018.index.month)==set(range(1,9)), 'Jan–ago deve existir em ambos os anos.'
olist_growth = (m_2018['receita'].sum()/m_2017['receita'].sum()-1)*100
olist_orders_growth = (m_2018['pedidos'].sum()/m_2017['pedidos'].sum()-1)*100
comparativo = pd.DataFrame({'Indicador':['Olist | GMV itens entregues (jan–ago)','Mercado | faturamento e-commerce (jan–dez)'], 'Crescimento_2018_vs_2017_pct':[olist_growth,market_growth], 'Janela':['jan-ago 2018 / jan-ago 2017','ano completo 2018 / 2017'], 'Comparabilidade':['GMV amostral, sem frete','Faturamento setorial - definição distinta']})
display(comparativo)
fig,ax=plt.subplots(figsize=(12,5))
bar=ax.barh(comparativo['Indicador'],comparativo['Crescimento_2018_vs_2017_pct'],color=[TEAL,BLUE],height=.53)
ax.bar_label(bar,fmt='%.1f%%',padding=5,fontweight='bold')
ax.set_xlim(0,max(comparativo['Crescimento_2018_vs_2017_pct'])*1.24)
ax.set_title('Olist cresceu em ritmo elevado; comparabilidade é limitada',loc='left',color=NAVY)
ax.set_xlabel('Variação (%) 2018 ante 2017 | períodos distintos')
ax.text(0,-.32,'Olist: jan–ago; setor: ano cheio. Métricas e universos diferentes.',transform=ax.transAxes,fontsize=10,color=GREY)
plt.tight_layout();save(fig,'09_contexto_setor_webshoppers')
print(f'Olist, jan–ago: GMV {olist_growth:+.1f}%; pedidos {olist_orders_growth:+.1f}%. Setor, ano completo: faturamento {market_growth:+.1f}%.')

## 9. Expansão regional | Intensidade relativa por habitante (IBGE 2018)

**Nova base:** estimativa populacional IBGE (UF, referência em 01/07/2018), arquivo oficial reeditado em 2022 para o ano-base 2018. O comparativo regional usa **apenas GMV de pedidos comprados em 2018 (jan–ago)**, não todo o período, dividido pela população estimada da UF em 2018. O índice de presença é: `(participação da UF no GMV de 2018) / (participação da UF na população brasileira de 2018)`. A interpretação indica intensidade relativa **na amostra**, não quota de mercado nem potencial de retorno; depende da cobertura e da composição da base.

Fonte: https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2018/POP2018_20220905.pdf (população por UF; soma nacional 208.494.900).

In [ ]:
# EXTENSÃO EXTERNA: população e indicadores por UF
pop = pd.read_csv(ROOT/'data'/'external'/'ibge_populacao_uf_2018.csv',dtype={'uf':'string'})
assert pop['uf'].nunique()==27 and pop['populacao'].sum()==208494900, 'Arquivo populacional não passou na validação.'
base18 = base.loc[(base['mes'].dt.year==2018)&(base['mes'].dt.month<=8)]
uf18 = base18.groupby('uf_cliente',as_index=False).agg(gmv_2018=('price','sum'),pedidos_2018=('order_id','nunique'))
regional = pop[['uf','populacao']].merge(uf18,left_on='uf',right_on='uf_cliente',how='left',validate='one_to_one')
regional[['gmv_2018','pedidos_2018']]=regional[['gmv_2018','pedidos_2018']].fillna(0)
regional['participacao_gmv']=regional['gmv_2018']/regional['gmv_2018'].sum()
regional['participacao_pop']=regional['populacao']/regional['populacao'].sum()
regional['indice_presenca']=regional['participacao_gmv']/regional['participacao_pop']
regional['gmv_por_habitante']=regional['gmv_2018']/regional['populacao']
regional=regional.sort_values('gmv_2018',ascending=False)
display(regional[['uf','gmv_2018','populacao','participacao_gmv','participacao_pop','indice_presenca','gmv_por_habitante']].head(15).style.format({'gmv_2018':brl,'populacao':'{:,.0f}','participacao_gmv':pct,'participacao_pop':pct,'indice_presenca':'{:.2f}','gmv_por_habitante':brl}))
# Comparação limitada às 12 UFs com maior volume financeiro para reduzir ruído de bases diminutas.
top_regions=regional.head(12).sort_values('indice_presenca')
fig,ax=plt.subplots(figsize=(12,7))
bars=ax.barh(top_regions['uf'],top_regions['indice_presenca'],color=[TEAL if x>=1 else BLUE for x in top_regions['indice_presenca']])
ax.axvline(1,color=ORANGE,ls='--',lw=2,label='Paridade com peso populacional')
ax.bar_label(bars,fmt='%.2f',padding=4)
ax.set_xlim(0,max(1.2,top_regions['indice_presenca'].max()*1.2))
ax.set_xlabel('Índice de presença relativa | amostra Olist, jan–ago/2018')
ax.set_title('Onde o GMV supera o peso demográfico?',loc='left',color=NAVY)
ax.legend(loc='lower right');plt.tight_layout();save(fig,'10_indice_presenca_regional')
fig,ax=plt.subplots(figsize=(11,6))
r=regional.query('gmv_2018>0')
ax.scatter(r['populacao']/1e6,r['gmv_2018']/1e6,s=90,alpha=.7,color=BLUE)
for _,row in r.sort_values('gmv_2018',ascending=False).head(8).iterrows():
    ax.annotate(row['uf'],(row['populacao']/1e6,row['gmv_2018']/1e6),xytext=(5,4),textcoords='offset points',fontsize=10)
ax.set_xlabel('População estimada em 2018 (milhões)');ax.set_ylabel('GMV jan–ago/2018 (R$ milhões)')
ax.set_title('Tamanho do mercado e presença observada',loc='left',color=NAVY)
plt.tight_layout();save(fig,'11_populacao_vs_gmv')

## 10. Síntese integrada | Decisões, limites e exportação

A nova etapa identifica **oportunidades de investigação, não causalidade**. Estados de presença relativa baixa merecem checagem de logística, cobertura, renda e aquisição; presença elevada pode refletir vantagem de distribuição e/ou viés da amostra. Não compare diretamente GMV de itens com receita setorial nem generalize estes números para a empresa atual.

In [ ]:
# EXTENSÃO EXTERNA: exportação consolidada, mantém contratos de saída antigos
# Atualiza o resumo executivo com contexto externo e critérios explícitos.
regional_top = regional.sort_values('indice_presenca',ascending=False).iloc[0]
addon = f"""
## Contexto externo | Ebit/Nielsen e IBGE
- **GMV Olist (jan–ago/2018 contra jan–ago/2017):** {olist_growth:+.1f}%.
- **Pedidos Olist (jan–ago/2018 contra jan–ago/2017):** {olist_orders_growth:+.1f}%.
- **Faturamento do e-commerce brasileiro (2018 contra 2017, anos completos):** {market_growth:+.1f}% (Webshoppers/Ebit-Nielsen).
- **Índice regional de presença relativa (2018):** maior valor em {regional_top['uf']}, {regional_top['indice_presenca']:.2f}; índice = share do GMV da amostra / share populacional do IBGE.

### Implicações de decisão
1. Validar capacidade de manter crescimento de pedidos sem deterioração de margem e serviço.
2. Executar pilotos de aquisição e logística em UFs de menor presença relativa, considerando tamanho econômico e custo de atendimento.
3. Priorizar categorias e sellers com retorno comprovado, mitigando concentração e monitorando custos.

### Cuidados metodológicos adicionais
- **Não** interpretar a diferença entre taxas da Olist e do mercado como ganho de market share: métrica e janelas diferentes (8 meses versus 12 meses), amostra limitada.
- A população é do ano-base 2018 e o GMV regional usa apenas compras de janeiro a agosto de 2018.
- Índice de presença não é demanda potencial, participação de mercado ou previsão de retorno.
- Referências: https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2018/POP2018_20220905.pdf ; https://www.terra.com.br/byte/receita-do-e-commerce-no-brasil-foi-de-r-532-bi-em-2018-diz-estudo,586591935998801e529b01f0566a08fdt0ugp598.html
"""
(OUT/'resumo_executivo.md').write_text(summary+addon,encoding='utf-8')
with pd.ExcelWriter(OUT/'tabelas_analiticas.xlsx',engine='openpyxl') as writer:
    monthly.to_excel(writer,sheet_name='Mensal')
    cat.to_excel(writer,sheet_name='Categorias')
    uf.to_excel(writer,sheet_name='UF_Clientes')
    municipio.to_excel(writer,sheet_name='Municipios')
    uf_seller.to_excel(writer,sheet_name='UF_Sellers')
    prod.head(100).to_excel(writer,sheet_name='Top_100_Produtos')
    seller.head(100).to_excel(writer,sheet_name='Top_100_Sellers')
    market.to_excel(writer,sheet_name='Mercado_Ebit',index=False)
    comparativo.to_excel(writer,sheet_name='Comparativo_Externo',index=False)
    regional.to_excel(writer,sheet_name='IBGE_UF_Presenca',index=False)
print('Arquivos atualizados:', OUT/'resumo_executivo.md', OUT/'tabelas_analiticas.xlsx')
display(Markdown(addon))

## 11. Decomposição do crescimento: volume × ticket

**Objetivo executivo:** descobrir quanto da variação do GMV entre **janeiro–agosto de 2017** e **janeiro–agosto de 2018** decorreu do número de pedidos, quanto decorreu do ticket médio e qual foi a interação. Com as definições deste projeto, **GMV = pedidos entregues com itens × ticket médio**.

O waterfall decompõe uma identidade contábil da amostra, **não prova as causas econômicas do crescimento**. O intervalo 2018 jan–ago foi escolhido para evitar comparar um ano completo a um período parcial.

In [ ]:
# Análise:  ponte de crescimento auditável, sem estimativas externas
assert set(m_2017.index.month)==set(range(1,9)) and set(m_2018.index.month)==set(range(1,9))
q17, q18 = int(m_2017['pedidos'].sum()), int(m_2018['pedidos'].sum())
g17, g18 = float(m_2017['receita'].sum()), float(m_2018['receita'].sum())
t17, t18 = g17/q17, g18/q18
bridge = pd.DataFrame({
    'fator': ['Volume de pedidos', 'Ticket médio', 'Interação volume × ticket'],
    'efeito_gmv_brl': [(q18-q17)*t17, q17*(t18-t17), (q18-q17)*(t18-t17)]
})
assert np.isclose(bridge['efeito_gmv_brl'].sum(), g18-g17, atol=.02), 'A ponte não fecha no GMV.'
display(pd.DataFrame({'Indicador':['Pedidos','Ticket médio','GMV de produtos'],
                      '2017':[q17,t17,g17], '2018':[q18,t18,g18],
                      'Crescimento_pct':[(q18/q17-1)*100,(t18/t17-1)*100,(g18/g17-1)*100]}))
fig,ax=plt.subplots(figsize=(11,5))
colors=[BLUE,TEAL,ORANGE]
bars=ax.barh(bridge['fator'],bridge['efeito_gmv_brl']/1e6,color=colors,height=.55)
ax.axvline(0,color=GREY,lw=.7)
ax.bar_label(bars,labels=[f'{v/1e6:+.2f} mi'.replace('.',',') for v in bridge['efeito_gmv_brl']],padding=4)
ax.set_xlim(left=min(-.12,bridge['efeito_gmv_brl'].min()/1e6-.1),right=max(4,bridge['efeito_gmv_brl'].max()/1e6*1.18))
ax.set_xlabel('Efeito aritmético no GMV (R$ milhões)')
ax.set_title('A expansão do GMV veio predominantemente de mais pedidos',loc='left',color=NAVY)
plt.tight_layout();save(fig,'12_decomposicao_crescimento')
print('Conferência da ponte (R$):',brl(bridge.efeito_gmv_brl.sum()),'=',brl(g18-g17))

## 12. Cenários de crescimento e sensibilidade

**Escopo:** exercícios *hipotéticos de sensibilidade*, não previsão estatística nem orçamento. Referência: GMV de **jan–ago/2018**, com **a mesma extensão de oito meses**; as variações são aplicadas simultaneamente ao volume e ao ticket médio mantendo composição constante.

$$GMV_{cenario} = GMV_{base}\,(1+\Delta_{\mathrm{pedidos}})\,(1+\Delta_{\mathrm{ticket}})$$

Os percentuais escolhidos abaixo são **premissas didáticas ajustáveis**, não probabilidades. Nenhum valor calculado representa receita líquida, margem, EBITDA, ROI ou projeção de vendas reais.

In [ ]:
# Análise:  premissas editáveis. Rodar esta célula após ajustar parâmetros.
SCENARIOS = {
    'Conservador': {'delta_pedidos':0.05, 'delta_ticket':0.00},
    'Base':        {'delta_pedidos':0.10, 'delta_ticket':0.03},
    'Otimista':    {'delta_pedidos':0.15, 'delta_ticket':0.05},
}
cenarios = pd.DataFrame([{'cenario':nome, **v} for nome,v in SCENARIOS.items()])
assert ((cenarios[['delta_pedidos','delta_ticket']] > -1).all().all())
cenarios['gmv_base_brl'] = g18
cenarios['gmv_simulado_brl'] = g18*(1+cenarios['delta_pedidos'])*(1+cenarios['delta_ticket'])
cenarios['incremento_gmv_brl'] = cenarios['gmv_simulado_brl']-g18
cenarios['incremento_gmv_pct'] = 100*(cenarios['gmv_simulado_brl']/g18-1)
cenarios['natureza'] = 'Hipótese/sensibilidade; NÃO previsão'
display(cenarios.style.format({'delta_pedidos':pct,'delta_ticket':pct,'gmv_base_brl':brl,
                              'gmv_simulado_brl':brl,'incremento_gmv_brl':brl,'incremento_gmv_pct':'{:.1f}%'}))
fig,ax=plt.subplots(figsize=(11,5))
nomes=['Base observada']+cenarios['cenario'].tolist()
valores=[g18/1e6]+cenarios['gmv_simulado_brl'].div(1e6).tolist()
bars=ax.bar(nomes,valores,color=[GREY,BLUE,TEAL,ORANGE],width=.6)
ax.bar_label(bars,labels=[f'R$ {v:.2f} mi'.replace('.',',') for v in valores],padding=5,fontweight='bold')
ax.set_ylim(0,max(valores)*1.22)
ax.set_ylabel('GMV hipotético para uma janela equivalente de 8 meses (R$ mi)')
ax.set_title('O crescimento potencial depende de duas alavancas: pedidos e ticket',loc='left',color=NAVY)
plt.tight_layout();save(fig,'13_cenarios_crescimento')

In [ ]:
# Análise:  matriz de sensibilidade; variações deliberadamente amplas para identificar exposição
pedido_hipoteses = [-0.05,0,0.05,0.10,0.15,0.20]
ticket_hipoteses = [-0.03,0,0.03,0.05,0.08]
sensibilidade = pd.DataFrame({
    f'Ticket {t:+.0%}': [((1+p)*(1+t)-1)*100 for p in pedido_hipoteses]
    for t in ticket_hipoteses
},index=[f'Pedidos {p:+.0%}' for p in pedido_hipoteses])
fig,ax=plt.subplots(figsize=(12,6))
sns.heatmap(sensibilidade,annot=True,fmt='.1f',cmap='BrBG',center=0,linewidths=.5,
            cbar_kws={'label':'Variação hipotética do GMV (%)'},ax=ax)
ax.set_title('Sensibilidade do GMV a volume e ticket | % versus base',loc='left',color=NAVY)
ax.set_xlabel('Variação do ticket médio');ax.set_ylabel('Variação dos pedidos')
plt.tight_layout();save(fig,'14_matriz_sensibilidade')
display(sensibilidade.round(1))

## 13. Priorização regional com critérios explícitos

**Triagem, não previsão de mercado**: selecionamos estados que tenham (a) índice de presença inferior a 0,85, (b) população estimada ≥ 3 milhões e (c) pelo menos 500 pedidos entregues na amostra jan–ago/2018. Esses limiares são **regras de decisão ilustrativas e ajustáveis**, não limites estatísticos nem regras de negócio da Olist.

Entre os elegíveis, ordenamos pelo **déficit percentual de participação demográfica menos participação no GMV** (`gap_pp`, em pontos percentuais), sem monetizar esse gap. Isso identifica assimetrias para investigar; não estima demanda reprimida, valor adicional capturável ou retorno marginal. **Antes de investir**, avaliar custos de aquisição, disponibilidade de sellers, frete, prazos, conversão e margem — variáveis ausentes nas bases.

In [ ]:
# Análise:  filtros transparentes e auditáveis. Ajustáveis pelo avaliador.
MIN_POP = 3_000_000
MIN_PEDIDOS = 500
MAX_INDICE = 0.85
priorizacao = regional[['uf','populacao','pedidos_2018','gmv_2018','participacao_gmv',
                        'participacao_pop','indice_presenca']].copy()
priorizacao['gap_pp'] = 100*(priorizacao['participacao_pop'] - priorizacao['participacao_gmv'])
priorizacao['elegivel_piloto'] = ((priorizacao['populacao']>=MIN_POP) &
    (priorizacao['pedidos_2018']>=MIN_PEDIDOS) & (priorizacao['indice_presenca']<MAX_INDICE))
priorizacao['status'] = np.where(priorizacao['elegivel_piloto'],'Candidata à investigação','Sem prioridade pelos filtros')
priorizacao=priorizacao.sort_values(['elegivel_piloto','gap_pp'],ascending=[False,False])
pilotos=priorizacao.loc[priorizacao['elegivel_piloto']].copy().sort_values('gap_pp',ascending=False)
assert not pilotos.empty, 'Os filtros configurados não geraram nenhuma UF para análise.'
display(pilotos[['uf','populacao','pedidos_2018','indice_presenca','gap_pp']].style.format({
  'populacao':'{:,.0f}','pedidos_2018':'{:,.0f}','indice_presenca':'{:.2f}','gap_pp':'{:.2f} p.p.'}))
fig,ax=plt.subplots(figsize=(11,5))
r=pilotos.sort_values('gap_pp')
bars=ax.barh(r['uf'],r['gap_pp'],color=TEAL,height=.6)
ax.bar_label(bars,labels=[f'{v:.2f} p.p.'.replace('.',',') for v in r['gap_pp']],padding=5)
ax.set_xlim(0,r['gap_pp'].max()*1.25)
ax.set_xlabel('Participação da população − participação no GMV de 2018 (p.p.)')
ax.set_title('UFs para diagnóstico: presença proporcionalmente menor',loc='left',color=NAVY)
plt.tight_layout();save(fig,'15_triagem_regional')
print('Atenção: ranking exploratório sem dados de logística, CAC, margem ou potencial de retorno.')

In [ ]:
# Análise:  robustez da triagem frente a diferentes parâmetros (evita ranking "preciso" ilusório)
parametros = [(3_000_000,300,0.85),(3_000_000,500,0.85),(3_000_000,800,0.85),
              (5_000_000,500,0.85),(3_000_000,500,0.70)]
robustez=[]
for pop_min,ped_min,limite in parametros:
    lista=priorizacao.loc[(priorizacao['populacao']>=pop_min) &
                          (priorizacao['pedidos_2018']>=ped_min) &
                          (priorizacao['indice_presenca']<limite)].sort_values('gap_pp',ascending=False)
    robustez.append({'pop_min':pop_min,'pedidos_min':ped_min,'indice_max':limite,
                     'top_ufs':', '.join(lista.head(5)['uf']), 'qtd_elegiveis':len(lista)})
robustez=pd.DataFrame(robustez)
display(robustez)
print('UFs não listadas NÃO foram avaliadas como economicamente inviáveis; apenas não passaram os filtros.')

## 14. Plano de decisão: 90 dias, KPIs e validação

**Tese executiva:** o crescimento histórico da amostra foi movido principalmente por pedidos; recomenda-se fortalecer categorias/sellers relevantes e autorizar **pilotos regionais condicionados** — nunca investimento em escala automática com base em índices demográficos.

| Período | Ação executiva | Evidência para seguir | Regra de decisão |
|---|---|---|---|
| 0–30 dias | Instrumentar funil regional e baseline; auditar margem, estoque, frete e qualidade de serviço | Pedidos, GMV, margem de contribuição, conversão, custo de aquisição, cancelamentos e SLA por UF | **Não começar escala** sem dados de custos e grupo de comparação |
| 31–60 dias | Executar pilotos pequenos e comparáveis nas UFs elegíveis | Efeito incremental de pedidos e margem, com grupo-controle ou janela comparável | **Prosseguir** apenas se margem incremental líquida for positiva e SLAs aceitáveis |
| 61–90 dias | Revisão pelo comitê de investimento | Ganho incremental sustentado, risco de concentração e capacidade logística | **Escalar, corrigir ou interromper** por resultado; valores-meta definidos pelo negócio |

As métricas de margem, CAC, conversão e prazo não constam neste dataset; o notebook indica **quais dados coletar**, sem atribuir valores inexistentes.

In [ ]:
# Análise:  plano monitorável, separa métricas calculadas de lacunas de informação
plano=pd.DataFrame([
 {'etapa':'0–30 dias','acao':'Auditar custos e criar controle','kpi':'Margem, CAC, conversão, SLA','disponivel_no_olist':'Não','criterio':'Qualidade mínima dos dados e desenho de teste'},
 {'etapa':'31–60 dias','acao':'Piloto controlado nas UFs elegíveis','kpi':'Pedidos e GMV incrementais + margem incremental','disponivel_no_olist':'Parcial (pedidos/GMV históricos)','criterio':'Margem incremental positiva e serviço adequado'},
 {'etapa':'61–90 dias','acao':'Revisão de investimento','kpi':'Rentabilidade incremental e estabilidade da operação','disponivel_no_olist':'Não','criterio':'Escalar/corrigir/interromper com evidências'},
])
display(plano)

In [ ]:
# Análise:  exporta resultados novos, preservando TODOS os arquivos e abas existentes
from pathlib import Path
path_xlsx=OUT/'tabelas_analiticas.xlsx'
assert path_xlsx.exists(), 'Execute as células anteriores primeiro para gerar a planilha de métricas.'
with pd.ExcelWriter(path_xlsx,engine='openpyxl',mode='a',if_sheet_exists='replace') as writer:
    bridge.to_excel(writer,sheet_name='Decomposicao',index=False)
    cenarios.to_excel(writer,sheet_name='Cenarios',index=False)
    sensibilidade.to_excel(writer,sheet_name='Sensibilidade')
    priorizacao.to_excel(writer,sheet_name='Priorizacao_UF',index=False)
    robustez.to_excel(writer,sheet_name='Robustez',index=False)
    plano.to_excel(writer,sheet_name='Plano_90d',index=False)
cenarios.to_csv(OUT/'cenarios_crescimento.csv',index=False,encoding='utf-8-sig')
priorizacao.to_csv(OUT/'priorizacao_uf.csv',index=False,encoding='utf-8-sig')
# Evita duplicação ao reexecutar só esta célula.
summary_path=OUT/'resumo_executivo.md'
pre=summary_path.read_text(encoding='utf-8').split('## Alavancas de crescimento e decisões condicionais')[0].rstrip()
pilotos_txt=', '.join(pilotos.head(5)['uf'].tolist())
scenario_lines='\n'.join(f'- **{r.cenario}:** pedidos {r.delta_pedidos:+.0%}, ticket {r.delta_ticket:+.0%} ⇒ {brl(r.gmv_simulado_brl)} de GMV hipotético ({r.incremento_gmv_pct:+.1f}% vs. jan–ago/2018).' for r in cenarios.itertuples())
addon_decisao=f"""

## Alavancas de crescimento e decisões condicionais
- **Jan–ago/2017:** {q17:,} pedidos, {brl(g17)} de GMV e {brl(t17)} de ticket.
- **Jan–ago/2018:** {q18:,} pedidos, {brl(g18)} de GMV e {brl(t18)} de ticket.
- **Ponte do GMV:** efeito volume {brl(bridge.loc[0, 'efeito_gmv_brl'])}; efeito ticket {brl(bridge.loc[1,'efeito_gmv_brl'])}; interação {brl(bridge.loc[2,'efeito_gmv_brl'])}.

### Cenários hipotéticos em uma janela equivalente de 8 meses (NÃO previsões)
{scenario_lines}

### Priorização regional (triagem ilustrativa)
- **UFs elegíveis:** {pilotos_txt}. Critérios: população ≥ {MIN_POP:,}, pedidos ≥ {MIN_PEDIDOS}, índice de presença < {MAX_INDICE}. Ranking por diferença de participação da população e do GMV em pontos percentuais.
- **Não representa demanda não atendida, market share, ROI, causalidade ou retorno previsto.** Os resultados mudam conforme os limites configurados; ver planilha `Robustez`.

### Recomendação ao comitê
Executar testes controlados e escalonar investimentos **somente** após comprovar margem de contribuição incremental positiva, custos de aquisição aceitáveis e qualidade do serviço, dados indisponíveis na amostra. Reavaliar em 30, 60 e 90 dias.

### Rastreabilidade dos resultados
- Abas: Decomposicao, Cenarios, Sensibilidade, Priorizacao_UF, Robustez, Plano_90d.
- Fonte interna: Olist (pedidos entregues, itens); fontes externas: IBGE (população UF 2018) e Ebit/Nielsen (faturamento do e-commerce brasileiro em 2017 e 2018).
- Cenários = sensibilidade sobre o GMV jan–ago/2018; **não há previsão estatística** nem dados de margem/CAC.
"""
summary_path.write_text(pre+addon_decisao,encoding='utf-8')
print('✅ Resultados salvos: notebook, 4 gráficos adicionais, Excel com 6 abas de decisão, CSVs e resumo executivo.')
print('Cenários:', OUT/'cenarios_crescimento.csv')
print('Ranking:', OUT/'priorizacao_uf.csv')
display(Markdown(addon_decisao))